# FactLedger extractor

`load(path) -> documents, units` over raw files and nothing else. The extractor sniffs the
format from the bytes and writes document and unit JSON in the shapes of SCHEMA.md; the
rules it follows are in BUILD.md. Built one block at a time. Inputs: the public raw dataset
and the private papers dataset, both attached to this notebook.


In [ ]:
# Block 1: inputs and integrity.
# Mount both datasets, count files per folder, and check every file's sha256 against the
# folder manifest. The manifests are used here only to prove the Kaggle copies are the bytes
# that were uploaded; the extractor itself never reads them.
import hashlib
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

def mount(slug):
    """Kaggle mounts inputs at /kaggle/input/<slug> or, in newer sessions,
    /kaggle/input/datasets/<owner>/<slug>. Take whichever exists."""
    for candidate in (Path("/kaggle/input") / slug, Path("/kaggle/input/datasets/jhffmn") / slug):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(slug)


RAW = mount("it494-narrative-corpora-raw")
PAPERS = mount("it494-reference-papers")


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def check(folder, rows_key):
    manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
    rows = manifest[rows_key]
    on_disk = {p.name for p in folder.iterdir() if p.name not in ("manifest.json", "LICENSE")}
    listed = {r["file"] for r in rows}
    # Kaggle inputs are a network filesystem: one file at a time, 19,206 files take tens of
    # minutes; 32 concurrent reads take about a minute.
    with ThreadPoolExecutor(max_workers=32) as pool:
        digests = list(pool.map(sha256, [folder / r["file"] for r in rows]))
    bad = [r["file"] for r, d in zip(rows, digests) if d != r["sha256"]]
    print(f"{folder.name:<28} files {len(on_disk):>6}  listed {len(listed):>6}"
          f"  mismatched {len(bad)}  unlisted {len(on_disk - listed)}  missing {len(listed - on_disk)}")
    return bad


# The three literature manifests keep their original "works" key; the unpacked folders
# and the papers use "files".
for name, key in [("oz", "works"), ("holmes", "works"), ("greek", "works"),
                  ("graphrag-bench", "files"), ("longmemeval", "files")]:
    check(RAW / name, key)
check(PAPERS, "files")


In [ ]:
# Block 2: file type, then raw text.
#
# Two steps, by bytes only. Nothing here decides what the text is about; that is the model's
# job later.
#   1. file_kind(data): look at the first bytes and name the container: pdf, json, or text.
#   2. to_text(path): turn the container into one string, the document text.
#        pdf  -> the text layer, page by page (PyMuPDF, the one dependency)
#        json -> if it holds chat turns, one "role: content" block per turn under a header
#                of the session id and dates. We also keep where each turn starts and ends
#                in that string, so a chat can be cut into units without a model.
#        text -> the bytes decoded as UTF-8, unchanged
import json

try:
    import pymupdf
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf"], check=True)
    import pymupdf


def file_kind(data):
    if data.startswith(b"%PDF-"):
        return "pdf"
    if data.lstrip()[:1] in (b"{", b"["):
        return "json"
    return "text"


def pdf_text(data):
    doc = pymupdf.open(stream=data, filetype="pdf")
    return "\n".join(page.get_text() for page in doc)


def chat_turns(obj):
    """The list of {role, content} turns inside a chat JSON, wherever it sits; None if absent."""
    if isinstance(obj, list) and obj and all(isinstance(t, dict) and "role" in t and "content" in t for t in obj):
        return obj
    if isinstance(obj, dict):
        for value in obj.values():
            found = chat_turns(value)
            if found:
                return found
    return None


def chat_text(obj, turns):
    """Header lines, a blank line, then 'role: content' per turn. Returns the text and the
    (start, end) of each turn inside it."""
    header = [f"session_id: {obj['session_id']}"] if "session_id" in obj else []
    header += [f"date: {d}" for d in obj.get("dates", [])]
    text = "\n".join(header) + "\n\n"
    spans = []
    for turn in turns:
        start = len(text)
        text += f"{turn['role']}: {turn['content']}\n\n"
        spans.append((start, len(text)))
    return text, spans


def to_text(path):
    data = path.read_bytes()
    doc = {"path": str(path), "sha256": hashlib.sha256(data).hexdigest(),
           "kind": file_kind(data), "text": "", "turns": None, "dates": []}
    if doc["kind"] == "pdf":
        doc["text"] = pdf_text(data)
    elif doc["kind"] == "json":
        obj = json.loads(data)
        turns = chat_turns(obj)
        if turns is None:
            doc["kind"], doc["text"] = "text", data.decode("utf-8", errors="replace")
        else:
            doc["kind"] = "chat"
            doc["text"], doc["turns"] = chat_text(obj, turns)
            doc["dates"] = list(obj.get("dates", []))
    else:
        doc["text"] = data.decode("utf-8", errors="replace")
    return doc


# One of each, to see the shape.
for path in [RAW / "oz" / "01_55.txt", RAW / "graphrag-bench" / "Novel-30752.txt",
             RAW / "longmemeval" / "sharegpt_yywfIrx_0.json", RAW / "longmemeval" / "001cefa7_2.json",
             PAPERS / "edge2024-graphrag.pdf"]:
    d = to_text(path)
    turns = len(d["turns"]) if d["turns"] else "-"
    print(f"{path.name:<26} {d['kind']:<5} {len(d['text']):>8,} chars  turns {turns:>3}  dates {d['dates']}")
    print("    " + repr(d["text"][:70]))


In [ ]:
# Block 3: the model call.
import json
import re
import time

import requests
from kaggle_secrets import UserSecretsClient

MODEL = "gpt-5.6-luna"
RETRY = "gpt-5.6-terra"
PRICE = {"gpt-5.6-luna": (0.20, 1.20), "gpt-5.6-terra": (2.00, 12.00)}   # $ per M tokens in, out
SPEND_STOP = 8.00                                                        # dollars; the run halts past this
KEY = UserSecretsClient().get_secret("OPENAI_API_KEY")
calls = []


def spend():
    return sum(c["cost"] for c in calls)


def generate(prompt, model=MODEL, effort="low"):
    """One JSON-mode call; the reply parsed, the cost logged. The API rejects temperature."""
    if spend() >= SPEND_STOP:
        raise RuntimeError(f"spending stop: ${spend():.2f}")
    t0 = time.time()
    r = requests.post("https://api.openai.com/v1/chat/completions",
                      headers={"Authorization": f"Bearer {KEY}"}, timeout=300,
                      json={"model": model, "reasoning_effort": effort,
                            "response_format": {"type": "json_object"},
                            "messages": [{"role": "user", "content": prompt}]})
    if r.status_code != 200:
        raise RuntimeError(f"OpenAI {r.status_code}: {r.text}")
    body = r.json()
    u, (p_in, p_out) = body["usage"], PRICE[model]
    calls.append({"model": body["model"], "in": u["prompt_tokens"], "out": u["completion_tokens"],
                  "seconds": round(time.time() - t0, 1),
                  "cost": (u["prompt_tokens"] * p_in + u["completion_tokens"] * p_out) / 1e6})
    return json.loads(body["choices"][0]["message"]["content"])


print(f"model {MODEL}, retry {RETRY}, spend stop ${SPEND_STOP:.2f}, key {'present' if KEY else 'MISSING'}")


In [ ]:
# Block 5: the split call. The text is numbered by row (one row per line, or per sentence
# when the text has no line breaks), sent in slices, and the model answers with row numbers.
# A row number is an address: code turns it into a character offset by lookup.
SLICE = 200_000      # characters per call
CAP_WORDS = 4000

PROMPT = """Below is part %d of %d of one document. Each row is one line of the text (or one sentence, when the text has no line breaks), numbered. Answer with JSON only, pointing at places by row number.

{
  "source_class": one of "canonical" (a published literary or classic work), "published" (a paper, article, or report), "authored" (a person's own material: notes, email, letters, drafts); null unless this is part 1,
  "title": the title as written, or null,
  "author": the author's name as written, or null,
  "date": {"row": the row containing the date the work was written, published, or sent, "iso": "YYYY" or "YYYY-MM" or "YYYY-MM-DD"} or null. Not a transcription or ebook release date,
  "body_start": the row where the work itself begins, or null if it does not begin in this part. Publisher notices, a contents list, and transcriber's or translator's notes are not part of the work; an author's own preface or introduction is,
  "end_matter_start": the row where end matter begins after the work (license, index, notes, advertisements), or null if none begins in this part,
  "toc_count": the number of pieces a contents list gives, or null,
  "pieces": [{"row": the row of the heading where a piece begins, in the body itself and never in a contents list, "title": a short title for the piece, such as "Chapter 1: The Cyclone" or "Abstract" or "Act II, Scene 1"}]
}

Pieces are the document's own divisions: chapters, acts and scenes, sections, dated entries, poems, stories. A paper's pieces are its sections, the abstract first. Aim for pieces under %d words; where a division is longer, use its next level down. A part with no piece beginning in it gets an empty pieces list.

TEXT:
%s
"""

FIRST = ("source_class", "title", "author", "date", "body_start", "toc_count")   # first answer wins
LAST = ("end_matter_start",)                                                  # last answer wins


def rows(text):
    """[(offset, text)] per row: one per line, or one per sentence when the text has no lines."""
    if text.count("\n") >= len(text) / 500:
        starts = [0] + [m.end() for m in re.finditer(r"\n", text)]
    else:
        starts = [0] + [m.end() for m in re.finditer(r'(?<=[.!?])\s+(?=[A-Z"“])', text)]
    return [(s, text[s:e]) for s, e in zip(starts, starts[1:] + [len(text)])]


def slices(numbered):
    out, current, size = [], [], 0
    for line in numbered:
        if size + len(line) > SLICE and current:
            out.append("\n".join(current))
            current, size = [], 0
        current.append(line)
        size += len(line) + 1
    out.append("\n".join(current))
    return out


def propose(doc, model=MODEL):
    """One merged reply for the document, from the same question asked of every slice."""
    numbered = [f"{n}: {t.rstrip()}" for n, (offset, t) in enumerate(doc["rows"])]
    parts = slices(numbered)
    reply = {"pieces": []}
    for i, part in enumerate(parts):
        r = generate(PROMPT % (i + 1, len(parts), CAP_WORDS, part), model=model)
        for key in FIRST:
            if reply.get(key) is None and r.get(key) is not None:
                reply[key] = r[key]
        for key in LAST:
            if r.get(key) is not None:
                reply[key] = r[key]
        reply["pieces"] += r.get("pieces") or []
    return reply


In [ ]:
# Block 6: row numbers to character positions, over several documents. A row that does not
# exist, or rows out of order, mean the reply is unusable.


def breaks_for(doc, reply):
    n = len(doc["rows"])
    valid = [p["row"] for p in reply["pieces"] if isinstance(p.get("row"), int) and 0 <= p["row"] < n]
    bad = len(reply["pieces"]) - len(valid)
    found = [doc["rows"][r][0] for r in valid]
    ordered = all(a < b for a, b in zip(found, found[1:]))
    end = reply.get("end_matter_start")
    end = doc["rows"][end][0] if isinstance(end, int) and 0 <= end < n else len(doc["text"])
    return found, end, bad, ordered


for path in [RAW / "oz" / "01_55.txt", RAW / "oz" / "02_54.txt", RAW / "holmes" / "03_1661.txt",
             RAW / "greek" / "18_10523.txt", RAW / "greek" / "03_348.txt", RAW / "greek" / "27_library00apolgoog.txt",
             RAW / "graphrag-bench" / "Novel-30752.txt", PAPERS / "edge2024-graphrag.pdf"]:
    doc = to_text(path)
    doc["rows"] = rows(doc["text"])
    reply = propose(doc)
    found, end, bad, ordered = breaks_for(doc, reply)
    first = doc["text"][found[0]:found[0] + 30].strip() if found else "-"
    print(f"{path.name:<28} pieces {len(reply['pieces']):>3}  bad rows {bad:>2}  in order {str(ordered):<5}"
          f"  toc {reply.get('toc_count')!s:>4}  end {end}/{len(doc['text'])}  date {(reply.get('date') or {}).get('iso') or '-':<8}"
          f"  first piece at {found[0] if found else '-'}: {first!r}")
print(f"${spend():.3f} spent")
